In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. DOMAIN AND FLOW SETTINGS
# ------------------------------------------------------------

# Number of lattice nodes in the horizontal direction.
nx = 400

# Number of lattice nodes in the vertical direction.
ny = 120

# Relaxation time.
# Must be greater than 0.5 for positive viscosity.
tau = 0.6

# Horizontal inlet velocity in lattice units.
# Keep this small: LBM is a low-Mach-number method.
u_inlet = 0.04

# Number of collision-streaming time steps to run.
num_steps = 3000


# ------------------------------------------------------------
# 2. D2Q9 DISCRETE VELOCITIES AND WEIGHTS
# ------------------------------------------------------------

# Each row is one discrete velocity c_i = [cx, cy].
# The direction order must match w and every f[:, :, i].
c = np.array([
    [ 0,  0],  # 0: rest
    [ 1,  0],  # 1: east
    [ 0,  1],  # 2: north
    [-1,  0],  # 3: west
    [ 0, -1],  # 4: south
    [ 1,  1],  # 5: north-east
    [-1,  1],  # 6: north-west
    [-1, -1],  # 7: south-west
    [ 1, -1],  # 8: south-east
])

# D2Q9 equilibrium weights.
w = np.array([
    4 / 9,
    1 / 9, 1 / 9, 1 / 9, 1 / 9,
    1 / 36, 1 / 36, 1 / 36, 1 / 36,
])

# For D2Q9 with dx = dt = 1, cs^2 = 1/3.
cs_squared = 1 / 3


# ------------------------------------------------------------
# 3. CREATE A SOLID CYLINDER MASK
# ------------------------------------------------------------

# Make arrays containing every x- and y-location in the domain.
y, x = np.indices((ny, nx))

# Cylinder centre in lattice-node coordinates.
cylinder_x = nx // 4
cylinder_y = ny // 2

# Cylinder radius in lattice nodes.
radius = ny // 10

# True means solid; False means fluid.
solid = (x - cylinder_x)**2 + (y - cylinder_y)**2 <= radius**2
# basically creates a void in the shape of a cylinder in the middle of the domain, where the fluid cannot flow through. The rest of the domain is fluid.
# so when we call solid next time it will first check if the point is inside the cylinder and if it is it will return True and if not it will return False. This is used to set the initial velocity of the fluid to zero inside the cylinder and to set the initial density of the fluid to 1 everywhere else.



# ------------------------------------------------------------
# 4. INITIALISE MACROSCOPIC FLOW VARIABLES
# ------------------------------------------------------------

# Begin with uniform density everywhere.
rho = np.ones((ny, nx))

# Start with zero velocity in both directions.
ux = np.zeros((ny, nx))
uy = np.zeros((ny, nx))

# Prescribe initial rightward flow everywhere in the fluid. except solid
ux[~solid] = u_inlet
# fo any node wich is not solid, we set the x-velocity to u_inlet. This means that the fluid will start flowing to the right at a speed of u_inlet. The y-velocity is still zero everywhere, so the fluid will only flow in the x-direction.


# ------------------------------------------------------------
# 5. FUNCTION: BUILD EQUILIBRIUM POPULATIONS
# ------------------------------------------------------------

def equilibrium(rho, ux, uy):
    """
    Convert density and velocity fields into the nine D2Q9
    equilibrium population fields.

    Output shape: (ny, nx, 9)
    """

    # Create an empty array for f_eq.
    f_eq = np.zeros((ny, nx, 9))

    # Calculate u dot u = ux^2 + uy^2 at every lattice node.
    u_squared = ux**2 + uy**2

    # Calculate f_eq separately for each D2Q9 direction.
    for i in range(9):

        # c_dot_u means c_i dot u at every lattice node.
        c_dot_u = c[i, 0] * ux + c[i, 1] * uy

        # Full second-order D2Q9 equilibrium distribution.
        f_eq[:, :, i] = w[i] * rho * (
            1
            + c_dot_u / cs_squared
            + (c_dot_u**2) / (2 * cs_squared**2)
            - u_squared / (2 * cs_squared)
        )

    return f_eq


# Convert the initial rho, ux, uy fields into nine populations.
f = equilibrium(rho, ux, uy)


# ------------------------------------------------------------
# 6. OPPOSITE-DIRECTION MAP FOR BOUNCE-BACK
# ------------------------------------------------------------

# Example: east (1) reflects to west (3).
# North-east (5) reflects to south-west (7).
opposite = np.array([0, 3, 4, 1, 2, 7, 8, 5, 6])

In [ ]:
plt.figure(figsize=(12, 4))

plt.imshow(solid, origin="lower", cmap="gray_r")

plt.title("Cylinder geometry: white = fluid, black = solid")
plt.xlabel("x lattice node")
plt.ylabel("y lattice node")

plt.show()

In [ ]:
# ------------------------------------------------------------
# 7. TIME-ADVANCEMENT LOOP
# ------------------------------------------------------------

# Repeat one complete LBM time step num_steps times.
for step in range(num_steps):

    # --------------------------------------------------------
    # A. RECOVER MACROSCOPIC VARIABLES FROM THE POPULATIONS
    # --------------------------------------------------------

    # Add all nine populations at every lattice node.
    # Result shape: (ny, nx).
    rho = np.sum(f, axis=2)

    # Start x- and y-momentum fields at zero.
    momentum_x = np.zeros((ny, nx))
    momentum_y = np.zeros((ny, nx))

    # Add the momentum contribution from each D2Q9 direction.
    for i in range(9):

        # Add f_i multiplied by the x-component of c_i.
        momentum_x = momentum_x + f[:, :, i] * c[i, 0]

        # Add f_i multiplied by the y-component of c_i.
        momentum_y = momentum_y + f[:, :, i] * c[i, 1]

    # Divide momentum by density to obtain the velocity fields.
    ux = momentum_x / rho
    uy = momentum_y / rho

    # Solid nodes are not fluid, so define their macroscopic velocity as zero.
    ux[solid] = 0.0
    uy[solid] = 0.0


    # --------------------------------------------------------
    # B. COLLISION: RELAX f TOWARD ITS LOCAL EQUILIBRIUM
    # --------------------------------------------------------

    # Build f_eq using each node's current density and velocity.
    f_eq = equilibrium(rho, ux, uy)

    # Apply the BGK collision equation:
    # f_post_collision = f - (f - f_eq) / tau
    f_post_collision = f - (f - f_eq) / tau


    # --------------------------------------------------------
    # C. BOUNCE-BACK AT THE CYLINDER
    # --------------------------------------------------------

    # Select all nine post-collision populations at solid nodes.
    # [:, opposite] reverses every direction:
    # east becomes west, north becomes south, and so on.
    #
    # This is the simple full-way bounce-back wall condition.
    f_post_collision[solid, :] = f_post_collision[solid, :][:, opposite]


    # --------------------------------------------------------
    # D. STREAMING: MOVE EACH POPULATION TO A NEIGHBOUR
    # --------------------------------------------------------

    # Create an empty array for populations after streaming.
    f_streamed = np.zeros_like(f)

    # Stream each of the nine directional population fields.
    for i in range(9):

        # np.roll shifts values across a 2D array.
        #
        # axis=0 is the NumPy row direction, which increases downward.
        # Our D2Q9 convention defines +y as north/upward.
        # Therefore we use -c[i, 1] for the row shift.
        #
        # axis=1 is the x-direction, which increases rightward.
        # Therefore we use +c[i, 0] for the column shift.
        #
        # Example for east, c[1] = [1, 0]:
        # shift = (0, +1), so f1 moves one column right.
        f_streamed[:, :, i] = np.roll(
            f_post_collision[:, :, i],
            shift=(-c[i, 1], c[i, 0]),
            axis=(0, 1),
        )


    # --------------------------------------------------------
    # E. SIMPLE INLET AND OUTLET CONDITIONS
    # --------------------------------------------------------

    # Prescribe density = 1 and velocity = [u_inlet, 0]
    # at every node on the left boundary.
    rho_inlet = np.ones((ny, 1))
    ux_inlet = np.full((ny, 1), u_inlet)
    uy_inlet = np.zeros((ny, 1))

    # Convert the prescribed inlet macroscopic state into
    # its nine equilibrium populations.
    f_inlet = equilibrium(rho_inlet, ux_inlet, uy_inlet)

    # Replace all populations at x = 0 with the inlet populations.
    f_streamed[:, 0, :] = f_inlet[:, 0, :]

    # Use a simple zero-gradient outlet:
    # copy the second-last column into the final column.
    f_streamed[:, -1, :] = f_streamed[:, -2, :]

    # Use streamed populations as the state for the next time step.
    f = f_streamed


    # --------------------------------------------------------
    # F. OPTIONAL PROGRESS MESSAGE
    # --------------------------------------------------------

    # Print progress every 500 iterations.
    if step % 500 == 0:
        print(f"Completed step {step} of {num_steps}")

In [ ]:
# ------------------------------------------------------------
# 8. PLOT THE SPEED FIELD AFTER THE SIMULATION
# ------------------------------------------------------------

# Recover density from the final populations.
rho = np.sum(f, axis=2)

# Recover final x-momentum and y-momentum.
momentum_x = np.zeros((ny, nx))
momentum_y = np.zeros((ny, nx))

for i in range(9):
    momentum_x = momentum_x + f[:, :, i] * c[i, 0]
    momentum_y = momentum_y + f[:, :, i] * c[i, 1]

# Recover final velocity components.
ux = momentum_x / rho
uy = momentum_y / rho

# Calculate speed magnitude.
speed = np.sqrt(ux**2 + uy**2)

# Hide the cylinder interior from the colour plot.
speed[solid] = np.nan

# Create the figure.
plt.figure(figsize=(14, 4))

# Plot speed over the full domain.
plt.imshow(
    speed,
    origin="lower",
    cmap="turbo",
    aspect="auto",
)

# Add a colour bar with its physical meaning.
plt.colorbar(label="Speed magnitude |u| (lattice units)")

# Draw the cylinder edge.
plt.contour(
    solid,
    levels=[0.5],
    colors="black",
    linewidths=1.0,
    origin="lower",
)

# Label the plot.
plt.title("D2Q9 LBM: flow past a circular cylinder")
plt.xlabel("x lattice node")
plt.ylabel("y lattice node")

plt.show()